# Compare YOLOv7, YOLOv10 and YOLO26 on the EdgeAI Station

**Author:** Antoine ([@antowan](https://github.com/antowan))

Three detectors with the same job (`car`, `person`, `truck`) are a good test of what makes a model a good fit for an edge station:
*how accurate is it?* and *how fast does it run on this GPU?* This notebook reads the measurements recorded in the repository and
shows them side by side. **Reading it needs no station and no GPU**; the last section shows how to measure again on your own station.

| Flavor | Model | Build notebook |
|---|---|---|
| `yolov7` | YOLOv7-tiny | [`01-build-yolov7.ipynb`](01-build-yolov7.ipynb) |
| `yolov10` | YOLOv10n | [`02-build-yolov10.ipynb`](02-build-yolov10.ipynb) |
| `yolo26` | YOLO26n | [`03-build-yolo26.ipynb`](03-build-yolo26.ipynb) |

### What is compared

- **The model's accuracy** on the 5,000 labelled images of COCO val2017, with the metric Ultralytics uses for its own model tables.
- **The latency of the raw model** on the station's GPU, one request at a time, through ONNX Runtime and TensorRT.
- **The effect of FP16**: how much accuracy the faster half-precision engine gives up.

It does **not** measure pipeline throughput (decode, tracking and callbacks across cameras; use `eais-tools check perf`) or accuracy on your own scenes.
All three models use the publishers' official COCO weights; none of them was trained for this repository.

## 1. The metric: mAP 50:95

A detector returns boxes with a class and a confidence score. To score it, each detection is matched to a labelled (*ground-truth*) box of the same class:
it is a hit when the boxes overlap enough, measured as **IoU** (intersection over union: shared area divided by combined area).

| Name | Definition | What it rewards |
|---|---|---|
| **AP 50** | Average precision (the area under the precision-recall curve) when a hit needs IoU ≥ 0.50 | Finding the object, even with a loose box |
| **mAP 50:95** | AP averaged over the ten thresholds 0.50, 0.55, …, 0.95, then over the classes | Finding it **and** drawing a tight box |

**mAP 50:95 is the COCO primary metric** and the one in Ultralytics' model tables (*mAP<sup>val</sup> 50-95*, COCO val2017, 640 px).
`tools/accuracy.py` computes it with `pycocotools`, the reference COCO implementation, on the same 5,000 images, so the measured values are comparable to the published ones.

There are two views of the same engine:

- **80 classes**: the standard benchmark. Use it to compare with published numbers.
- **car, person, truck**: only the three classes this AI Processor reports. Detections of other classes are dropped before scoring.

### Why the earlier 12.6% was low

Earlier versions of this repository reported **12.64% mAP 50:95** on COCO. The evaluation was correct; the weights were not suited to the benchmark:
they had been fine-tuned for three classes on other data, and such weights transfer poorly to COCO. The official COCO weights used now
reproduce the figures published by the model authors, which is how we know the evaluation is right.

## 2. Load the recorded results

The measurements live in [`doc/benchmarks/orin-nx-16gb-jp6.0/`](../doc/benchmarks/orin-nx-16gb-jp6.0/): two JSON files per flavor
(`<flavor>-latency.json` from `tools/benchmark.py`, `<flavor>-accuracy.json` from `tools/accuracy.py`) with the versions, file hashes and
GPU conditions of the run. `tools/compare_models.py` turns them into the tables below. Nothing is typed in by hand.

In [ ]:
import os
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
assert Path("tools/compare_models.py").is_file(), "Open this notebook from the repository (notebooks/ folder)"

sys.path.insert(0, "tools")
import compare_models

# Point this to tmp/benchmarks to look at your own measurements (section 7).
RESULTS = Path("doc/benchmarks/orin-nx-16gb-jp6.0")
entries = compare_models.collect(RESULTS)
print("Measured:", ", ".join(entry["flavor"]["display_name"] for entry in entries))
print(compare_models.environment(entries))

## 3. Accuracy

Values are percentages, for the TensorRT FP16 engine (the one that runs in the pipeline) at batch 8. The *published* column is the
figure the authors give for the same weights (the end-to-end column for YOLO26n). The YOLOv7 authors give no comparable figure for the *tiny* model (only at 416 px, or for a different SiLU variant at 640 px).

In [ ]:
display(Markdown(compare_models.accuracy_table(entries)))

## 4. Speed

Median latency of one inference request on the station's GPU, in milliseconds **per image** (the batch time divided by the batch size), for the raw model
with the same measurement method on every runtime (see `00-presentation.ipynb` for the method). *img/s* is the throughput of one request after another;
*ORT* is the speed-up over ONNX Runtime with CUDA in FP32 on the same graph.

In [ ]:
display(Markdown(compare_models.latency_table(entries)))
display(Markdown("**All runtimes and precisions** (median of one request, in milliseconds):"))
display(Markdown(compare_models.precision_table(entries)))

## 5. Side by side

Left: speed per batch size (lower is better). Right: measured accuracy next to the published one (higher is better).

In [ ]:
display(Image(filename=str(RESULTS / "comparison.png")))

## 6. What FP16 costs in accuracy

FP16 halves the arithmetic precision of the engine. If it did hurt detection, this table would show it: the same images are scored with the FP32 and the FP16 engine.

In [ ]:
display(Markdown(compare_models.parity_table(entries)))

### Reading the results

- **Accuracy closely agrees with the published references.** The measured 80-class mAP 50:95 is within 0.2 points of the published figures for YOLO26n (40.1, the Ultralytics end-to-end column) and YOLOv10n (38.5, from the YOLOv10 paper), measured on the station's TensorRT FP16 engines. This is close agreement, not an exact reproduction of the authors' inference setup or a validation of the DeepStream pipeline. Ultralytics publishes 4.13 ms for YOLO26n FP16 on an Orin NX 16GB, but matching runtime, load and timing conditions have not been established. Its Jetson guide uses COCO128 for accuracy, not full COCO val2017; see the [measurement conditions](../doc/benchmarks/orin-nx-16gb-jp6.0/conditions.md).
- **YOLO26n has the highest recorded accuracy and the lowest measured latency of the three.** It scores 39.9 against 38.3 for YOLOv10n and 37.1 for YOLOv7-tiny. In the idle latency rerun it needs 5.3 ms per image at batch 1 against 5.5 and 8.4. YOLOv7-tiny takes 1.6x as long as YOLO26n at batch 1 and 1.4x at batch 8. YOLOv10n sits close to YOLO26n in speed (3.8 against 3.4 ms at batch 8).
- **The three classes score a little higher than the 80** (40.7 against 39.9 for YOLO26n), mostly thanks to `person`: per class, YOLO26n reaches 52.7 on `person`, 37.0 on `car` and 32.3 on `truck`. Trucks are the hardest of the three and the least frequent in the validation set (414 annotations against 1,918 cars and 10,777 persons).
- **The earlier accuracy run found negligible FP16 loss.** Scoring its FP32 and FP16 engines on the same images differs by at most 0.02 points. In the separate idle latency rerun, FP16 is 2.9 to 3.6 times faster than ONNX Runtime with FP32 at batch 1, depending on the model.
- **Batching amortizes the cost.** YOLO26n needs 3.4 ms per image at batch 8 against 5.3 ms at batch 1, so a batch of frames, one per stream, is cheaper per image than one frame at a time.
- **These are idle-baseline measurements, not pipeline throughput.** All 100 GPU-load baseline samples were 0%, with power mode and clock settings unchanged. Alternating-engine GPU-only checks kept the same ordering. Batch-1 latency is 30 to 38% lower than in the earlier contended run, but the engines were also rebuilt, so the difference cannot be attributed solely to stopping the pipeline. Accuracy values retain the earlier evaluation; the rebuilt engines were not rescored. See the [conditions](../doc/benchmarks/orin-nx-16gb-jp6.0/conditions.md).

**Choosing.** Start with **YOLO26n**: it is the most accurate and the fastest here, and it needs no NMS step. **YOLOv10n** is the same kind of model one generation earlier, a useful second reference. **YOLOv7-tiny** is the oldest and the slowest of the three, but it is the only one whose engine already contains the NMS step, which is a good example of the other way to design the output. The weights licenses also differ (GPL-3.0 for YOLOv7, AGPL-3.0 for the two Ultralytics-based models): read them before you distribute a package.

## 7. Measure again on your own station

This part is **optional** and needs a station with a GPU: run it in the station's Jupyter environment (App-Builder) from `/data/aip/AIPV1EMI0001`.
It exports the three models, builds the raw engines, times them and scores them on COCO val2017, which took **40 to 60 minutes per flavor** on the station used for the recorded
results (plus the one-time COCO download), mostly the TensorRT builds and the 5,000-image evaluation. Stop the video pipelines first: they share the GPU and would disturb the timings.
Set `MEASURE = True` to run the cells below; they write to `tmp/benchmarks`, never to the recorded results.

In [ ]:
MEASURE = False          # set to True to run the cells below
LIMIT = 0                # score only the first LIMIT images (0 = all 5,000); use e.g. 200 for a quick check
FLAVORS = ["yolov7", "yolov10", "yolo26"]

os.environ.update(MEASURE=str(int(MEASURE)), LIMIT=str(LIMIT), FLAVORS=" ".join(FLAVORS))

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
mkdir -p tmp/coco && cd tmp/coco
[ -d val2017 ] || { curl -fsSLO http://images.cocodataset.org/zips/val2017.zip && unzip -q val2017.zip && rm val2017.zip; }
[ -f annotations/instances_val2017.json ] || { curl -fsSLO http://images.cocodataset.org/annotations/annotations_trainval2017.zip \
    && unzip -q -o annotations_trainval2017.zip annotations/instances_val2017.json && rm annotations_trainval2017.zip; }
python3 -m pip install --user -r ../../tools/requirements-accuracy.txt
python3 -m pip install --user --no-deps -r ../../tools/requirements-export.txt
ls val2017 | wc -l

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
for flavor in $FLAVORS; do
    python3 tools/export_model.py "$flavor"
done

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
for flavor in $FLAVORS; do
    python3 tools/measure.py "$flavor" --annotations tmp/coco/annotations/instances_val2017.json \
        --images tmp/coco/val2017 --limit "$LIMIT" --results tmp/benchmarks
done

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
python3 tools/compare_models.py tmp/benchmarks

Then set `RESULTS = Path("tmp/benchmarks")` in section 2 and run the notebook again to see your own tables.

Expect small differences from the recorded numbers: latency depends on the station, its power mode and what else it is running.
Accuracy depends on the weights and the evaluation, and should match to a few hundredths of a point.